# 02 - Train a DQN Model Offline

This notebook shows the offline training workflow in Mouse Core:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and an action-value head.
4. Train the model with `DqnObjective` and save the checkpoint with `push_model_to_hub`.

The dataset comes from the collection notebook, but the same Mouse Core pieces apply to any sequential environment data stored as step dictionaries.


In [1]:
import torch

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"   # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                        # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                  # vocabulary size for discrete observations
GRADIENT_STEPS = 20000                 # total optimizer updates
SEQUENCE_LENGTH = 512                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                         # sequences per optimizer step


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(DATASET_ID, split="train", force_download=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

Stores 10 of 50:
Datastore(name='proc_frozenlake_0', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_1', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_10', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_11', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_12', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datastore(name='proc_frozenlake_13', steps=30000, columns=['action', 'step_index', 'reward', 'done', 'episode_index', 'task_index', 'observation', 'info_q_star'])
Datasto

## DataLoader And Augmentation

`DataLoader` samples sequence batches from one or more datastores. With `pack=True`, it can combine shorter segments while marking artificial boundaries so objectives can ignore transitions that cross packed segments.

`next_batch()` returns `list[list[dict]]` with shape `[batch][sequence]`: one list of step dictionaries per sampled sequence.

`Augmenter` can transform fields as batches are sampled. A modality spec names a row field and an augmentation type, such as `discrete`, `linear`, or `image`. Use `keep_fields` for values that should pass through unchanged because an objective still needs them.


In [3]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    # Preserve the pack-mode boundary marker used by objectives.
    keep_fields=["action", "observation", "reward", "done", "is_seam"],
)

loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    prefetch=4,
    num_workers=0,
    pack=True,
    augmenter=augmenter,
)

## Build The Model

A Mouse Core `Model` has three main pieces:

- `StepEmbedder` converts fields from each step dictionary into model tokens.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`StepEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and done codes.
- `rff` for scalar numeric values such as rewards.
- `learnable` when you want extra learned tokens that are not tied to a row field.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=10

## Train

The offline training loop is intentionally small because the Mouse Core abstractions do most of the work:

1. `loader.next_batch()` samples step sequences.
2. `model(batch)` embeds the dictionaries, runs the backbone, and produces head predictions.
3. `objective(objective_data, predictions)` computes the DQN loss and metrics.
4. The optimizer updates model weights.
5. `model.polyak_update(...)` updates target-network weights used by the objective.

`DqnObjective` interprets the integer `done` field through separate discount factors for normal steps, episode boundaries, and task boundaries. Set those gammas to match the semantics of your environment data.


In [5]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8
)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    tau=0.0005,
)

for step in range(GRADIENT_STEPS):

    batch = loader.next_batch()

    predictions, objective_data, _ = model(batch)
    loss, metrics = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")

loader.close()

/home/user/Repos/mouse-core/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


step=0  loss=0.0762  q=-0.004
step=100  loss=0.0515  q=0.772
step=200  loss=0.0186  q=0.608
step=300  loss=0.0186  q=0.483
step=400  loss=0.0161  q=0.653
step=500  loss=0.0226  q=0.687
step=600  loss=0.0132  q=0.574
step=700  loss=0.0090  q=0.514
step=800  loss=0.0156  q=0.591
step=900  loss=0.0298  q=0.462
step=1000  loss=0.0125  q=0.860
step=1100  loss=0.0131  q=0.734
step=1200  loss=0.0127  q=0.705
step=1300  loss=0.0126  q=0.824
step=1400  loss=0.0097  q=0.684
step=1500  loss=0.0112  q=0.702
step=1600  loss=0.0122  q=0.732
step=1700  loss=0.0101  q=0.612
step=1800  loss=0.0114  q=0.687
step=1900  loss=0.0117  q=0.772
step=2000  loss=0.0113  q=0.893
step=2100  loss=0.0100  q=0.757
step=2200  loss=0.0165  q=0.726
step=2300  loss=0.0103  q=0.850
step=2400  loss=0.0103  q=0.670
step=2500  loss=0.0124  q=0.799
step=2600  loss=0.0130  q=0.891
step=2700  loss=0.0107  q=0.952
step=2800  loss=0.0116  q=0.820
step=2900  loss=0.0081  q=0.972
step=3000  loss=0.0159  q=0.933
step=3100  loss=0.0

## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model-offline


: 